In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Session 6: Policy gradient II: advantage, PPO and GRPO

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - session 6: last session derived the Policy Gradient Theorem
    - this session makes it practical: baselines and advantages, then REINFORCE, PPO and GRPO
    - the next session applies them to language models
- **This session's goal**
    - see how the noisy raw-return weight becomes a stable update
- **By the end you can**
    - say why subtracting a baseline helps and does not bias the gradient
    - name the advantage used by REINFORCE, PPO and GRPO
    - explain the probability ratio and why clipping keeps the update small
    - read the PPO pseudo-code: actor, critic, old actor and reference model


# Policy gradient based methods: concepts

Recall the Policy Gradient Theorem
- formulated with single reward $\rlret_{0,\tau}$ for the entire trajectory

$$
\nabla_\theta J(\theta) = 
\Exp{\tau \sim \pr{\theta}} { 
\sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta   \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) \, \,\rlret_{0,\tau}
}
} 
$$

- formulated with intermediate rewards

$$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) \, \disc^\tt \rlret_{\tau,\tt} \right]
$$

Policy based methods
update parameters $\theta$
- in the direction (gradient)
- that increases expected return 

# Advantage: Beyond Rewards

The Policy Gradient Theorem states that the gradient for the trajectory $\tau$ is the unweighted sum
- of per step gradients


But perhaps the policy should be 
- *reinforced (positively or negatively)*
- in different proportions ?


Suppose all returns are positive.
- the Policy Gradient Theorem would push up *every* sampled action
- in proportion to the raw return, even when most of that return would have been obtained anyway

Or suppose we always get "easy" questions correct and only sometimes get "hard" questions correct
- the Policy Gradient Theorem would positively reinforce the already correct easy questions
- with as much force as it positively reinforces the hard question
    - for methods that involve multiple trajectories per example (e.g., GRPO)


We need some way to *relativize* 
 the return for a state
- by comparing it to a *baseline* (the anticipated return) for the state.
    - often $\statevalfun_\pi ( \stateseq_{\tt})$, the value of the state as would be computed by a Value function
- we call this relativized value the *advantage*

We will use the advantage as a mechanism to differentially weight the gradient of individual actions.





For example: consider a baseline that is the average return from the state
- above average returns have positive advantage
- below average returns have a negative advantage

Thus, in the case of all returns being positive
- we favor actions that result in "above expected" return (positive advantages)
- we disfavor actions that result "below expected" return (negative advantages)

Subtracting a baseline from the return has other advantages
- reduces the *magnitude* of the parameter update
    - smoother training
- reduces "noise"
    - the baseline value for a state is common to *all actions* from the state
    - so the advantage is relative to the signal *only* from the action itself


| Aspect                    | Effect of Subtracting Baseline                         |
|:---------------------------|:-------------------------------------------------------|
| Noise cancellation        | Removes expected (common) return, reducing fluctuations |
| Bias introduction         | None, baseline does not depend on action              |
| Zero-centering            | Advantages centered around zero, stabilizing updates  |



## Credit Assignment: The Evolution of the Advantage

"Early" Policy based methods (e.g., REINFORCE) were a direct expression of the Policy Gradient Theorem


$$
\nabla_\theta J(\theta) = 
\Exp{\tau \sim \pr{\theta}} { 
\sum_{\tt=0}^{|\tau|-1} {
\nabla_\theta   \log \pi(\actseq_{\tau,\tt} | \stateseq_{\tau,\tt}) \, \,\rlret_{0,\tau}
}
} 
$$

where the gradient of each step $\tt$ was equally weighted ( by the episode return ($\rlret_{0,\tau}$).



The problem with this
- some steps are "good": would have led to high episode return
- some steps are "bad":  decreased the potential episode return

But, with sparse rewards-
- we don't know how to *assign credit* for individual steps

For example:

- If an LLM writes a 100-word masterpiece 
- but makes a typo on the very last character
- the final reward might be low. 

Equal weighting of gradients punishes *all* steps whereas a single step is clearly to blame.

By obfuscating good signals from bad signals, learning is slowed
- high variance

Computing an Advantage per step
- is a means of solving the *Credit Assignment* problem
- creating differential degrees of reinforcement

"Modern RL" is associated with a pivot away from Reward and toward Advantage.
- [Asynchronous Advantage Actor-Critic](https://arxiv.org/abs/1602.01783) 2016

Having a separate Advantage $\advseq_{\tau,\tt}$ for each step $\tt$
- dictates the "Force" and "Direction" of the gradient weight update

| Advantage Value | Effect on Gradient | The "Steering" Result |
| :--- | :--- | :--- |
| **Positive ($\advseq_{\tau,\tt} > 0$)** | Multiplies by a positive scalar. | **Accelerate:** This action was better than the "usual." Make it more likely. |
| **Negative ($\advseq_{\tau,\tt} < 0$)** | Flips the sign of the gradient. | **Brake:** This action fell below expectations. Make it less likely. |
| **Zero ($\advseq_{\tau,\tt} \approx 0$)** | Nullifies the gradient. | **Maintain:** You did exactly what was expected. No learning required. |

Good steps are encouraged; bad (e.g., below expectation) steps are discouraged

The Policy Based methods we will study take different approaches to *Credit Assignment* (computing Advantage).


- REINFORCE (episode-return form)
    - equal credit to every step (the return-to-go form weights steps by the rewards that follow)
- PPO (Surgical Credit)
    - assigns credit to each step
    - by maintaining a prediction of the value of each state (a Value function)
- GRPO (Statistical Credit)
    - Uses the **group mean** of the returns of multiple completions as the baseline, and divides by their **standard deviation** 
    - It gives the **same Advantage** to every token in a specific trajectory. 
    - Each completion is one episode $\tau_i$, $i = 1, \ldots, G$: its single reward (written $r_i$ in the papers) is the episode return $\rlret_{0,\tau_i}$, the same quantity as in the *Episode Return form* of the theorem
    - The advantage of completion $i$ is $\left( \rlret_{0,\tau_i} - \operatorname{mean}(\rlret_{0,\tau_1}, \ldots, \rlret_{0,\tau_G}) \right) / \operatorname{std}(\rlret_{0,\tau_1}, \ldots, \rlret_{0,\tau_G})$, written with $\operatorname{mean}$ and $\operatorname{std}$ as in the papers
    - **Notation note:** the GRPO papers (DeepSeekMath, DeepSeek-R1, DAPO) write the *group size* as $G$. The return in these notebooks, $\rlret_\tt$ or $\rlret_{0,\tau}$, is also written $G$ in the literature. We keep both conventions: the group size is a bare count ($i = 1, \ldots, G$) while the return always carries a subscript, so context tells them apart


---

## Unified Policy Gradient Formulation: Advantage Definitions

Even though different Policy Based methods compute Advantage differently
- we can uniformly implement weighted Policy Gradient updates
- by rewriting the Policy Gradient Theorem in terms of Advantage

$$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) \advseq_{\tau,\tt} \right]
$$

**Note:** as in most implementations, the factor $\disc^\tt$ of the exact discounted gradient is absorbed into the weight $\advseq_{\tau,\tt}$ (it is $1$ when $\disc = 1$).

This allows us to focus on the *Advantage definition* 
$$
\advseq_{\tau,\tt} 
$$
as a way of differentiating methods.

We thus refer to the above formulation as the Unified Policy Gradient Formulation.

The policy-based methods we present will typically be variations of this "vanilla" gradient.

# Policy gradient based methods: Preview

We will show several common policy-based methods.
- REINFORCE
- PPO (the "classic" method)
- GRPO (relatively new: 2024)

Details of each follow.

As a preview, we show the definition of the Advantage for each.


| Method  | Intermediate Reward Advantage $\advseq_{\tau, \tt}$      | Single Trajectory Reward Advantage $\advseq_\tau$         |
|:---------|:--------------------------------------------|:--------------------------------------------------|
| REINFORCE | $\advseq_\tt = \rlret_\tt - b_\tt$                        | $\advseq = \rlret_{0,\tau} - b$                              |
| PPO     | $\advseq_\tt = \hat{\advseq}_\tt$ (e.g., GAE estimator) | $\advseq = \rlret_{0,\tau} - b$ applied per step             |
| GRPO    | Relative advantages per candidate           | Same per-candidate advantage applied per step    |


And a comparison:

| Method | Gradient-Based | Main Objective | Key Characteristics | Stability & Sample Efficiency | Typical Application Domains |
|:--------|:----------------|:----------------|:---------------------|:------------------------------|:-----------------------------|
| **PPO** (Proximal Policy Optimization) | Yes | Maximize expected reward with clipped surrogate objective | Uses policy gradients with clipping to limit policy update magnitude, balancing exploration and exploitation | High stability; more sample efficient than vanilla policy gradients; widely used for continuous and discrete control tasks | Robotics, games, continuous control, benchmark RL tasks |
| **GRPO** (Group Relative Policy Optimization) | Yes | Optimize policy using group-relative advantage estimates | Does not require a separate value function; updates policy based on relative advantages within a group of candidate outputs | More memory efficient, stable; effective for large-scale policy optimization with reduced critic reliance | Training large language models, large-scale policy optimization |
| **REINFORCE** | Yes | Maximize expected cumulative reward by direct policy gradient | Uses Monte Carlo sampled returns, applies likelihood ratio trick; pure policy gradient without value function | High variance and sample inefficient; simpler but less stable than actor-critic or PPO | Fundamental policy gradient algorithm, baseline for many RL studies |


# REINFORCE (the "Original")

REINFORCE is an early method (historically)
that is very close to the Policy Gradient Theorem formulation
- uses Advantage rather than the episode Return
- the Advantage subtracts a "baseline" from the episode Return

| Method  | Intermediate Reward Advantage $\advseq_{\tau, \tt}$      | Single Trajectory Reward Advantage $\advseq_\tau$         |
|:---------|:--------------------------------------------|:--------------------------------------------------|
| REINFORCE | $\advseq_\tt = \rlret_\tt - b_\tt$                        | $\advseq = \rlret_{0,\tau} - b$                              |
|

The purpose of subtracting a baseline is to
- reduce the magnitude of the gradients
- reduces the variance of the gradients
- and hence: the change in policy parameters $\theta$

In  single,  end-of-episode reward, the Advantage is typically computed as
$$
\advseq_{\tau,\tt} = \left( \rlret_{0,\tau} - b \right)
$$

where $b$ is a baseline
- computed as the Global Moving Average of rewards
- over *all steps in the trajectory of all prompts*
- processed thus far


The Baseline Update Rule: 
- Every time you finish a rollout and get an episode return $\rlret_{0,\tau}$, 
- update the baseline:
$$b_{new} = (1 - w)\, b_{old} + w \, \rlret_{0,\tau}$$

where $w$ is a small weight of the moving average (not the learning rate $\alpha$)


The baseline calibrates the reward received for the current prompt/trajectory against all
previous prompts.

With intermediate rewards, the Advantage is typically computed as
$$
\advseq_{\tau,\tt} = \left( \rlret_{\tau, \tt} - b_\tt \right)
$$
- where $b_\tt$ is often a proxy for the *expected* Value function $\statevalfun_\pi(\stateseq_\tt)$ of state $\stateseq_\tt$
- requires a separate Critic to approximate the Value function

It uses "Monte-Carlo sampling" method to estimate the gradient

**Note**

"Monte Carlo sampling" is a confusing way of saying: complete trajectory for the single prompt

The word "Monte Carlo" is thus used in the same sense as 
"Monte Carlo" in the [Temporal Difference methods we studied](RL_Value_based_model_free.ipynb#Monte-Carlo-(MC)-method)

## Pseudo code for REINFORCE

    # REINFORCE training for LLM
    for prompt in training_prompts:
        output = llm.generate(prompt)
        reward = evaluate_output(output) # Human or automated score
        logprob = llm.logprob(output, prompt)

        # Monte Carlo policy gradient update (no critic)
        baseline = compute_baseline() # Optional: running mean for variance reduction
        loss = -logprob * (reward - baseline)
        loss.backward()
        optimizer.step()


## Discussion

REINFORCE is very close to the vanilla Policy Gradient.

But it is considered high variance
- due to a source not related to the baseline


The Advantage depends on
$$\rlret_{\tt, \tau}$$

which is computed *based on the single trajectory*  $\tau$ that was encountered.

Thus, we are observing the Environment's Transition Probabilities 
- via a *single sample* from its distribution, for each step

The weight is therefore the return of a *single sampled episode*, which depends on the Environment's random responses: it fluctuates from episode to episode (high variance).

Subsequent methods will attempt to mitigate this source of variance.


    
In the *practical* implementation of REINFORCE
- we estimate $\rlret_{\tt}$
- over a *batch* of episodes $\tau_1, \ldots, $ for the same prompt

so we have *multiple samples*  from the probability distribution for the State/Action pair $(\state, \act)$.
- hopefully leading to a lower variance approximation

The "practical" version below will be very similar to the GRPO method we subsequently introduce.

    # Practical RL Loop Logic
    for prompt in batch:
        # 1. Sample G trajectories for the SAME prompt
        trajectories = model.generate(prompt, n=8) 

        # 2. Get rewards
        rewards = reward_model(trajectories)

        # 3. Calculate Local Baseline (The "Modern" Way)
        # This cancels out prompt difficulty automatically
        baseline = rewards.mean() 

        # 4. Standardized Advantage
        advantages = (rewards - baseline) / (rewards.std() + 1e-8)

        # 5. Compute loss weighted by how much better/worse each sample was
        loss = -log_probs * advantages

### Comparison: Theoretical vs. Practical REINFORCE

| Feature | Theoretical REINFORCE | Practical (e.g., GRPO / Batch RL) |
| :--- | :--- | :--- |
| **Rollouts per Prompt** | **1** | **Multiple (G = 4 to 64)** |
| **Baseline Type** | **Global Moving Average** (Historical) | **Local Group Mean** (Immediate) |
| **Variance** | **Very High** (Requires small learning rates) | **Reduced** (Brute-forced by sampling) |
| **Update Frequency** | Every single trajectory | After a group/batch of trajectories |
| **Compute Cost** | Low per update | High per update (heavy generation) |
| **Analogy** | Checking your lifetime golf average. | Comparing 10 swings from the same spot. |

# Surrogate Loss

Rather than
- maximization of return

Policy methods often switch to
- minimization of a *surrogate loss*

The surrogate loss 
- supplements the (negative of) return
- with **constraints** on the derived policy and  on the solution process.

Whereas the  Policy Gradient Theorem (Return Maximization) provides
- the *theoretical* basis for an optimal policy
- using the Returns or Advantages

the Surrogate Loss provides
- the *practical* objective for finding the optimal policy
- subject to practical constraints: stability, convergence

The actual implementation of our models will
usually revert to the Minimization of Surrogate Loss formulation.

What are some constraints we typically encounter ?

We often see *limitations* on how far an updated policy can change
- prevents drastic policy shifts after a parameter update
- by constraining the new policy to be close to a "reference" policy

Other constraints try to promote stability and convergence in the solution process
- limiting the magnitude of a gradient update
- promoting low variance of the gradient updates

**Constraints**

| Reason                          | Explanation                                               |
|:---------------------------------|:-----------------------------------------------------------|
| Trust Region Constraints        | Avoid overly large policy updates that destabilize learning |
| Regularization                  | KL divergence or clipping adds penalty to maintain stability  |
| Stability & Convergence         | Ensures smooth, incremental improvement of the policy       |

**Relationship between Policy Gradient Theorem and Surrogate Loss**

| Aspect                  | Policy Gradient Theorem                                     | Surrogate Loss                                               |
|:------------------------|:------------------------------------------------------------|:-------------------------------------------------------------|
| Role                    | Theoretical formula for expected return gradient            | Practical approximation/objective for policy update          |
| Gradient                | Directly gives unbiased gradient of $J(\theta)$             | Its gradient approximates policy gradient but includes stabilizing terms |
| Constraints             | None intrinsic; pure gradient formula                        | Includes clipping, penalties to enforce trust region and avoid large steps |
| Use in Algorithms       | Foundation for policy gradient methods                       | Used in PPO-style algorithms to compute safe gradient steps   |
| Goal                    | Maximize expected return $J(\theta)$                         | Provide stable, incremental improvement proxy                |


# PPO (the "Classic")


## Intuition

PPO optimizes a **clipped surrogate** loss
- which can be interpreted as variant of the standard policy gradient theorem.

The goal of the surrogate objective is to control how rapidly the policy changes from iteration to iteration of training.
- avoid large policy shifts
- smaller, more stable updates
- (a monotonic-improvement guarantee belongs to the theory of TRPO, not to PPO's clipping)

## Surrogate Loss for PPO

The Surrogate Loss for PPO is
$$
J_{\text{PPO}}(\theta) = \E_\tt \left[
\min{} \left(
r_\tt(\theta) \hat{\advseq}_\tt,\;
\text{clip}\left(r_\tt(\theta), 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}\right) \hat{\advseq}_\tt
\right)
\right]
$$

We will explain how this potentially intimidating equation came about.

We express the 
- relative change in policy (for a given State/Action pair)
    - of the current Actor
    - compared to the Old Actor (Actor at start of current iteration)
- via the *probability ratio*
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\text{old}}}(\actseq_\tt |\stateseq_\tt)}
$$

where $\pi_{\theta_{\text{old}}}$ is the *Old Actor policy*

**Notation note:** the PPO paper writes the probability ratio as $r_\tt(\theta)$, and we keep it. This $r$ is not the reward (written $\rew$ or $\rewseq_{\tt+1}$ here); context tells them apart.


The Old Actor policy is the policy in effect at the start of each iteration of training
- before this iteration modifies the policy parameters
- it is the policy *under which examples for the current batch* have been generated
    - the *Behavior Policy* in On-Policy/Off-Policy terminology

During each iteration of training
- episodes/trajectories are generated  using the *Old Actor* policy
    - stored in the "experience buffer" (when PPO is applied in batches vs single examples)
    - are used *multiple times* during the iteration
- the policy parameters are updated based on the surrogate loss for these trajectories
- the updated policy becomes the Old Actor policy for the next iteration

By keeping the probability ratio close to $1$, we constrain the Gradient Descent update step to a small change in policy.


How do we keep the probability ratio close to $1$ ?
- by using clipping to constrain it to the range $[1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}} ]$ for small $\epsilon_{\text{clip}}$

this is the *clipped* part of the clipped surrogate objective.

The PPO paper writes the clip range as $\epsilon$; we add the subscript, $\epsilon_{\text{clip}}$, to distinguish it from the exploration probability $\epsilon$ of $\epsilon$-greedy methods.

Formally

The *surrogate loss* in PPO is:

$$
L_{\text{sur}}(\theta) = \E_\tt \left[ r_\tt(\theta) \hat{\advseq}_\tt \right] 
= \E_\tt \left[ \frac{\pi_\theta(\actseq_\tt|\stateseq_\tt)}{\pi_{\theta_{\text{old}}}(\actseq_\tt|\stateseq_\tt)} \hat{\advseq}_\tt \right]
$$

where

- $ r_\tt(\theta)$  is the probability ratio,
- $\hat{\advseq}_\tt$ is the advantage estimate at step $\tt$.



We modify the Surrogate Loss to limit the effect of the Probability Ratio.

The *clipped surrogate objective* (to be maximized) is

$$
L_{\text{clip}}(\theta) = \E_\tt \left[ \min{} \left( r_\tt(\theta) \hat{\advseq}_\tt, \; \text{clip}\left(r_\tt(\theta), 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}} \right) \hat{\advseq}_\tt \right) \right]
$$

We maximize it (Gradient Ascent), in a controlled manner
- equivalently, we minimize its negative, which is what the code below calls `L_clip`



**What clipping does**

<img src="images/ppo_clip.png" width=95%>

- $r > 1$ means the action became more likely than under the old actor; $r < 1$, less likely
- with a positive advantage the objective stops rewarding an increase of $r$ beyond $1 + \epsilon_{\text{clip}}$; with a negative advantage, a decrease below $1 - \epsilon_{\text{clip}}$
- where the curve is flat the gradient is zero, so the update stops pushing that action further


## Iterations vs Epochs

You may have noticed our introduction of an unfamiliar word: *iteration*

In Supervised Learning
- there is a fixed set of examples for the entirety of training
    - targets are fixed
- an *epoch* processes all the examples

In Reinforcement Learning
- training is based on episodes/trajectories
    - episodes/trajectories: the response to an initial state (e.g., "prompt" in LLM terms)
    - generated by an Actor policy
- the Actor policy gets updated during training (within each epoch)
    - thus, even though the "prompts" are fixed
    - *new episodes* are generated by an updated Actor
- so we have *iterations*
    - **episodes fixed** within an iteration
        - create at the start of the iteration
        - using the updated Actor
    - epochs process the *latest* set of examples
        - we repeat the epoch multiple times
 

Why do we repeat epochs multiple times within an iteration ?
 
- sample efficiency

Creating a long episode is expensive
- we want to learn as much as possible before discarding it


## Relation to the Unified Gradient Formulation

The (unclipped) Surrogate Loss does not resemble the  Unified Gradient Formulation.
- absence of the term

$$
\log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) 
$$

that multiplies the Advantage.

Moreover,
 there is an extra term
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\text{old}}}(\actseq_\tt |\stateseq_\tt)}
$$

in the form of the Probability Ratio.


But, these terms appear inside the Expectation of the Gradient and, by using the Likelihood Ratio trick
$$
\begin{array}{llll} \\
\nabla_\theta r_\tt(\theta) & = & r_\tt (\theta) \nabla_\theta \log r_\tt (\theta) & \text{Likelihood Ratio trick} \\
& = & r_\tt (\theta) \nabla_\theta \left( 
    \log \pi_\theta(\actseq_\tt|\stateseq_\tt) - \log \pi_{\text{old}} (\actseq_\tt|\stateseq_\tt) 
\right)  & \text{log of ratio converted to difference in logs} \\
& = & r_\tt (\theta) \nabla_\theta \log \pi_\theta(\actseq_\tt|\stateseq_\tt) & \text{since } \log \pi_{\text{old}} \text{ is not a function of } \theta \\
\end{array}
$$

Thus, taking the gradient of the Probability Ratio
- yields the usual
$$
\log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) 
$$

term.



We still have an extra $r_\tt (\theta)$ multiplicative term (relative to the Unified Gradient Formulation).

So we don't obtain an *identical* expression
- but we note that, with clipping, $r_\tt (\theta) \approx 1$

## Advantage for PPO

To complete the presentation in terms of the Unified Gradient formulation, we define the Advantage.

| Reward Setting         | Advantage Estimate                                  |
|:-----------------------|:-----------------------------------------------------|
| Intermediate Rewards  | $\hat{\advseq}_\tt = \rlret_\tt -  \statevalfun_\pi(\stateseq_\tt)$ or GAE                 |
| Single Total Reward   | $\hat{\advseq}_\tt = \rlret_{0,\tau} - b$   |



We need to do a "deeper dive" on PPO in order to truly understand (and appreciate)
the Advantage definition.

In short
- PPO uses a Critic NN to estimate a Value function.e.g., the
$$
\statevalfun_\pi(\stateseq_\tt)
$$
term in the table above

OR

the TD error (here called the *local surprise*)
$$
\delta_{\tt}
$$
in the GAE description below.


**Using Return-to-go and value estimate:**

$
\hat{\advseq}_\tt = \rlret_\tt - \statevalfun_\pi(\stateseq_\tt)
$

where 
- $\rlret_\tt$ is the *actual* return-to-go starting from time step $\tt$
    - in the trajectory that was encountered
- $\statevalfun_\pi(\stateseq_\tt)$ is the value of $\rlret_\tt$ that was anticipated by the Value function

That is $\hat{\advseq}_\tt$ 
- is a measure of the error between actual fact and previously assumed value.



We can use the TD error $\delta_\tt$ of the value-based sessions, here called the *local surprise*
- as an *attribution of the error*

to each step in the trajectory

$
\hat{\advseq}_\tt = \sum_{l=0}^{\infty} (\disc \lambda)^l \delta_{\tt+l}
$

where

$
\delta_\tt = \rewseq_{\tt+1} + \disc \statevalfun_\pi(\stateseq_{\tt+1}) - \statevalfun_\pi(\stateseq_\tt)
$

with

- $\disc$ as the discount factor,
- $\lambda \in [0, 1]$ controlling the bias-variance trade-off.

**Aside**

- When $\lambda = 0$, GAE reduces to the **1-step Temporal Difference (TD) advantage** $\hat{\advseq}_\tt = \delta_\tt$:

- When $\lambda = 1$, GAE reduces to the **Monte Carlo advantage** $\hat{\advseq}_\tt = \rlret_\tt - \statevalfun_\pi(\stateseq_\tt)$:

GAE smoothly interpolates between 
- high-bias/low-variance and low-bias/high-variance advantage estimates

making it very effective in practice.

## Pseudo code for PPO

There are many subtleties to the code, which we discuss below:

    # --- INITIALIZATION ---
    # pi_theta: Training Actor (weights: theta)
    # v_phi:    Training Critic (weights: phi)
    # pi_old:   Rollout Snapshot (weights: theta_old)
    # pi_ref:   Safety Anchor (weights: frozen)

    for iteration = 1, 2, ... do:

        # -------------------------------------------------------------------------
        # 1. ROLLOUT PHASE (Sampling)
        # -------------------------------------------------------------------------
        # ACTION BY: [ACTOR (pi_theta)]
        # We generate trajectories using the live model weights
        actions, states, rewards = Rollout(pi_theta)

        # -------------------------------------------------------------------------
        # 2. PRE-COMPUTATION (Frozen Stats for Optimization)
        # -------------------------------------------------------------------------
        with torch.no_grad():
            # CALCULATED BY: [OLD ACTOR (pi_old)]
            # Anchor the original action probabilities for the ratio calculation
            logp_old = pi_old(actions | states)

            # CALCULATED BY: [CRITIC (v_phi)]
            # Get baseline "expected rewards" to center the training signal
            values_old = v_phi(states)

            # CALCULATED BY: [REFERENCE (pi_ref)]
            # Establish the "standard" behavior probability to check for drift
            logp_ref = pi_ref(actions | states)

            # [ADVANTAGE CALCULATION]
            # Combines Rollout rewards (Env) and Critic predictions (Baseline)
            # Goal: Reduce Environment Variance
            A_hat = Compute_GAE(rewards, values_old)
            Returns = A_hat + values_old

        # -------------------------------------------------------------------------
        # 3. OPTIMIZATION PHASE (Learning)
        # -------------------------------------------------------------------------
        for epoch = 1 to K do:

            # CALCULATED BY: [ACTOR (pi_theta)]
            # Get logprobs of the SAME actions under CURRENTLY UPDATING weights
            logp_curr = pi_theta(actions | states)

            # CALCULATED BY: [CRITIC (v_phi)]
            # Get current value predictions to compare against actual Returns
            v_curr = v_phi(states)

            # --- LOSS CALCULATION ---

            # [ACTOR LOSS] - Uses pi_theta (logp_curr) & pi_old (logp_old) & Critic (A_hat)
            ratio = exp(logp_curr - logp_old)
            L_clip = -min(ratio * A_hat, clip(ratio, 1-eps, 1+eps) * A_hat)

            # [KL PENALTY] - Uses pi_theta (logp_curr) & pi_ref (logp_ref)
            L_kl = beta * (logp_curr - logp_ref)

            # [CRITIC LOSS] - Uses v_phi (v_curr) & actual trajectory Returns
            L_value = c1 * MSE(v_curr, Returns)

            # [UPDATE]
            # Calculate gradients for theta and phi
            Total_Loss = L_clip + L_kl + L_value
            Update theta, phi via SGD

        # -------------------------------------------------------------------------
        # 4. SYNCHRONIZATION
        # -------------------------------------------------------------------------
        # Update pi_old for the next iteration's ratio anchor
        pi_old.load_state_dict(pi_theta.state_dict())

### Key observations about the code

**Iterations vs epochs**

- Examples for the iteration are created at the start of the iteration
- The model that creates the examples
    - changes at the end of the iteration
    - **not** within the iteration
- the Actor and Critic are updated
    - **within each epoch** of **each iteration**
- the Actor **does not** create new episodes using the updated policy
    - it uses the episodes created at start of iteration
    - evaluates *new probabilities* on the states/actions of the start-of-iteration episode

**Multiple models are computing values: be aware of which does what**        

There is **one set of examples** per iteration
- ROLLOUT phase
    - creates examples that are **fixed** for this iteration
    - created by the Actor from end of previous iteration
        - this Actor is called *Old Actor* subsequent to the ROLLOUT phase
    - *Old Actor* is the **Behavior Policy** (in On-policy/Off-policy terminology)
      
- PRE-COMPUTATION phase
    - we now call the model that create the examples the OLD ACTOR
    - the Advantage is computed with respect to the OLD ACTOR
        - using rewards and states from the episodes (examples) created by OLD ACTOR
        - NOT the rewards, states generated by the current (updated) ACTOR model
        - Using the Critic as it was at the end of the previous *iteration*
            - (`values_old` is computed once per iteration, before the epoch loop)
            - to compute state values

- OPTIMIZATION phase
    - the episodes remain the ones generated by the OLD ACTOR
        - **not** using updated policy of the ACTOR
        - so the states/actions/rewards are the same
        - but the *probabilities* are updated
            - used in the ratio

**Loss function**

Contains multiple terms
- `L_clip` and `L_kl` are part of Actor Loss
    - updates Actor parameters $\theta$
- `L-value` are part of Critic Loss (MSE)
    - updates Critic parameters $\phi$
    
So both Actor and Critic
- are updated each *epoch*

The Old Actor
- changes at end of *iteration*
    - to the Actor model as updated at end of the iteration

## Trust Region Policy Optimization (TRPO) Theory (Brief)

Constraining the probability ratio
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\text{old}}}(\actseq_\tt |\stateseq_\tt)}
$$
to the small range $$[1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}]$$

- reduces the magnitude of Gradient  updates
- thus reducing the variance of Gradient updates
- (TRPO's theory gives a monotonic-improvement bound for its exact constrained problem; clipping is only a practical approximation of that idea)

Constraining the "Actor" versus "Old" policy is based on
 *Trust Region Policy Optimizaton (TRPO)*.

We state TRPO briefly.

TRPO 
- limits the change between two policies
- "Old" and "Actor" (current)

under the assumption that we "trust" the "Old"
- it is the Behavior Policy that generated the examples

It *constrains* the optimization
- to maximize a surrogate objective 

$$
\max{\theta} \quad \E_{\state,\act \sim \pi_{\theta_{\text{old}}}} \left[
    \frac{\pi_\theta(\act \mid \state)}{\pi_{\theta_{\text{old}}}(\act \mid \state)} \advseq^{\pi_{\theta_{\text{old}}}}(\state,\act)
\right]
$$
- subject to a constraint 
    - on how much the new policy can deviate from the old policy:
    $$
\E_{\state \sim \pi_{\theta_{\text{old}}}} \left[
    D_{\text{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \big)
\right] \leq \delta_{\text{KL}}
$$

where:

- $\pi_{\theta_{\text{old}}}$ is the old  policy,
- $\pi_\theta$ is the new policy parameterized by $\theta$,
- $\advseq^{\pi_{\theta_{\text{old}}}}(\state,\act)$ is the advantage function with respect to the old policy,
- $D_{\text{KL}}(\cdot \| \cdot)$ is the Kullback-Leibler (KL) divergence measuring the difference between the old and new policies,
- $\delta_{\text{KL}}$ is a small positive constant controlling the maximum allowed policy update
    - the TRPO paper writes this bound as $\delta$; the subscript distinguishes it from the TD error $\delta_\tt$

With regards to the "trust region":

TRPO
- uses the KL divergence term as a constraint

PPO
- uses the *clipping* of the ratio term as a constraint


PPO can be interpreted as a practical approximation of TRPO 
- that replaces the hard KL constraint
- with a clipped probability ratio 



**Technical note**

The first argument in KL term of  TRPO
- is $\pi_\text{old}$
- **not** $\pi_\theta$

The KL function is *asymmetric*
- the expectation is taken over $\pi_\text{old}$ not $\pi_\theta$

This is because "Old" is the behavior policy -- it is our "empirical distribution" of training
- any action introduced by the updated policy
- which had $0$ probability in "Old"
- *has no direct effect* on the KL term
    - since it has $0$ weight
    - only an indirect effect by shifting probability from elsewhere

# GRPO (the "new kid")

GRPO is a Policy Gradient based method that turns the rewards of a group of completions into *relative preferences*: a completion is preferred if its return is above the group mean and not preferred if it is below. It does not need preference pairs.

It will be presented in a separate module with other Preference based methods.

# What we established, and what it sets up

- **Established**
    - subtracting a baseline gives an advantage: the same gradient on average, less noise, and credit that can be positive or negative
    - REINFORCE weights by return minus a baseline; PPO by a critic-based advantage with a clipped probability ratio; GRPO by a group-relative advantage
    - the surrogate loss is the practical objective: it keeps each update close to the old policy
- **Sets up**
    - next session: the language model is the policy, a response is an episode, and GRPO needs no critic
    - the self-study appendix collects the bias and variance of the estimators


# Appendix (self-study): bias and variance of the estimators

- these slides compare the estimators met in the value-based sessions and here: Monte Carlo, TD, dynamic programming, Q-learning
- they are not presented in class (tagged skip) but stay in the notebook


# Bias and Variance

We now have multiple methods for RL
- how can we compare them ?

We will introduce the concepts of Bias and Variance as one metric with which to compare methods.

To be concrete:

What are the advantages/disadvantages of Temporal Difference(1 step) vs Monte Carlo in updating
the Value function ?


Consider iteration $k$:

In TD:
- the value of the successor state is *not* the true end-of-episode $k$ return
- it is an *estimate* of the state's value based on the *previous* iteration
- updating one estimate using another estimated is called *bootstrapping*

In MC
- the value of the successor state is the *true* end-of-episode $k$ return

Because TD depends on an estimate
- we say that TD is *biased*
    - since it is based only on the probabilities *sampled thus far*


Moreover, let's suppose that rewards are stochastic
- hence, each reward is a random variable

TD 1-step update 
- depends on **exactly one** random variable $$\hat\rew_{\tt+1}$$

TD k-step update
- depends on $k$ random variables: rewards for steps $\tt+1, \ldots, \tt+k$

MC update
- depends on **at least one** random variable
    - rewards of *all* subsequent states of the episode $\pi$
 $$\hat\rew_{\tt+1}, \ldots, $$

The variance of a sum of $k$ *independent* random variables is the sum of their variances
- *exactly* $k$ times the variance of each, if the variances are equal

So the variance of the $\statevalfun$ estimated by the various methods
- is smallest for TD 1-step
- increasing with $k$ for TD k-step
- is largest for Monte Carlo


**But**
- the *bias* decreases in the opposite direction of the increase of the variance
- estimate based on more information

Thus
- TD(0) is
    - low variance (one random variable), high bias (based on estimates)
- MC is
    - high variance (*at least* one random variable), **zero** bias (true values, not estimates)
    

## Bias and Variance of the methods presented

| Method                   | Bias          | Variance      | Notes                                                                                                        |
|:------------------------ |:------------- |:------------- |:------------------------------------------------------------------------------------------------------------ |
| DP (Dynamic Programming) | Low           | Low           | Uses full environment model; updates use true expectations.                                                  |
| MC (Monte Carlo)         | None          | High          | Unbiased (targets equal expected return), but episodes can have widely varying returns.                      |
| TD (Temporal Difference) | Moderate      | Moderate/Low  | Bootstrap introduces bias (approximate next value), but reduces variance compared to MC.                     |
| V-learning (State Value) | Moderate      | Low           | Like TD if bootstrapped; low-variance due to value averaging.                                                |
| Q-learning               | Moderate/High | Moderate/High | Off-policy, can have bias (maximization, bootstrapping); variance may increase due to noise in max operator. |

**Key Points**

DP (Dynamic Programming): Lowest bias and variance, but needs a full model (rarely available in practice).

MC: Unbiased estimates, but high variance due to sampling full returns.

TD: Bootstrapping introduces bias but greatly lowers variance versus MC.

V-learning: Similar profile to TD (if bootstrapped); often lower variance than Q-learning due to value averaging.

Q-learning: Can be more biased/variable due to off-policy backups and maximization over sampled estimates.



# Sources of Variance in RL

We introduced Variance as a way of comparing TD and MC for value-based methods.

But variance is present in many RL methods and is considered 
 a potential impediment to Learning via RL.
- bigger updates in
    - Value function for Value-based methods
    - Parameters for Policy-based methods
- can cause large changes in Policy
- which can lead to unstable training

We list the types of methods affected by each cause.

There multiple sources of Variance, which we summarize below
- stochastic environment
    - stochastic rewards and state transitions
- stochastic policy
- sparse rewards
    - per-episode vs per-step rewards
    - reward estimates become more noisy
- bootstrapping    


| Source of Variance             | Description                                                       | Methods Most Affected                |
|:-------------------------------|:-------------------------------------------------------------------|:--------------------------------------|
| Environmental Stochasticity   | Random rewards and transitions cause unpredictable outcomes        | Monte Carlo, TD, Policy Gradients    |
| Policy Stochasticity          | Probabilistic (non-deterministic) action selection                | MC, On-policy, Policy Gradients      |
| Long Trajectory Aggregation   | Returns summed over many steps compound the randomness             | Monte Carlo, n-step methods          |
| Sparse or Delayed Rewards     | Few positive signals lead to noisy estimates                       | Monte Carlo, Value-based             |
| Bootstrapping Error           | Using own predictions as targets introduces bias, but lowers variance | TD, Q-Learning                    |
| Credit Assignment Difficulty  | Uncertainty in linking actions to future rewards increases variance | Policy Gradients, MC                 |
| Sample Size & Exploration     | Insufficient samples or aggressive exploration cause wide fluctuations| All methods                       |
| Model/Optimization Instability| Neural network or optimizer issues amplify variance                | Deep RL, Policy Gradient methods     |
| Non-stationarity              | Changing environment or policy during training                     | All methods, esp. online learning    |

**Key Points:**  
- Variance comes from randomness in data, policy, training procedure, and modeling choices.  
- Some sources can be controlled with design choices (e.g., baselines, bootstrapping, averaging), while others are intrinsic to RL problems.

As we introduce new RL methods
- one motivation is to reduce variance

Some potential ways to reduce variance
- $n \gt 1$-step ahead Temporal Difference
- estimating updates in mini-batches
    - as in Gradient Descent


In [2]:
print("Done")

Done
